# 03 - Exploratory Data Analysis (EDA)

**Goal**: Visualize distributions, trends, and correlations in the cleaned dataset.
Answer initial research questions and identify patterns for ML modeling.

**Learning concepts**: Descriptive statistics, distribution analysis, correlation,
time series visualization, box plots for group comparison.

**Research themes explored**:
- Theme 1 (Salary): What do salary distributions look like across experience levels?
- Theme 2 (Ghost Jobs): How long do postings stay open? What's the distribution of `days_open`?
- Theme 3 (Entry-Level): What experience levels dominate the dataset?
- Theme 4 (Engagement): What drives the applies-to-views ratio?

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from talentlens.config import POSTINGS_CLEAN_PARQUET
from talentlens.plots import (
    plot_top_categories, plot_distribution, plot_box_by_category,
    plot_time_series, plot_correlation_heatmap, save_fig,
)

pd.set_option('display.max_columns', None)
%matplotlib inline

In [ ]:
# Load the cleaned dataset (fast from Parquet!)
df = pd.read_parquet(POSTINGS_CLEAN_PARQUET)
print(f"Loaded {len(df):,} cleaned postings")
print(f"Columns: {list(df.columns)}")

## 1. Job Title Distribution

What kinds of jobs dominate the dataset?

In [ ]:
plot_top_categories(
    df['title'],
    title='Top 25 Job Titles',
    top_n=25,
    save_name='eda_top_job_titles'
)
plt.show()

## 2. Experience Level Distribution

**Research Theme 3**: What proportion of jobs are labeled "Entry level"?

In [ ]:
print("Experience Level Counts:")
exp_counts = df['experience_level'].value_counts()
print(exp_counts)
print(f"\nPercentages:")
print((exp_counts / len(df) * 100).round(1))

plot_top_categories(
    df['experience_level'],
    title='Job Postings by Experience Level',
    top_n=10,
    save_name='eda_experience_levels'
)
plt.show()

## 3. Salary Distributions

**Research Theme 1**: What do salaries look like? How do they vary by experience level and remote status?

In [ ]:
# Filter to postings with salary data
df_salary = df[df['med_salary_yearly'].notna()].copy()
print(f"Postings with salary data: {len(df_salary):,} ({len(df_salary)/len(df)*100:.1f}%)")

# Overall salary distribution (clip outliers for visualization)
plot_distribution(
    df_salary['med_salary_yearly'],
    title='Distribution of Median Yearly Salary',
    xlabel='Median Salary (USD)',
    clip=(20_000, 300_000),
    bins=60,
    save_name='eda_salary_distribution'
)
plt.show()

print("\nSalary statistics:")
print(df_salary['med_salary_yearly'].describe().round(0))

In [ ]:
# Salary by experience level
exp_order = ['Internship', 'Entry level', 'Associate', 'Mid-Senior level', 'Director', 'Executive']
exp_order = [e for e in exp_order if e in df_salary['experience_level'].unique()]

plot_box_by_category(
    df_salary[df_salary['med_salary_yearly'].between(20_000, 400_000)],
    x='experience_level',
    y='med_salary_yearly',
    title='Salary Distribution by Experience Level',
    order=exp_order,
    save_name='eda_salary_by_experience'
)
plt.show()

In [ ]:
# Research Q3: Remote salary penalty/premium
remote_salary = df_salary.groupby('is_remote')['med_salary_yearly'].describe().round(0)
print("=== Salary by Remote Status ===")
print(remote_salary)

fig, ax = plt.subplots(figsize=(10, 6))
df_salary[df_salary['med_salary_yearly'].between(20_000, 400_000)].boxplot(
    column='med_salary_yearly',
    by='is_remote',
    ax=ax
)
ax.set_title('Salary: Remote vs On-Site')
ax.set_xlabel('Is Remote')
ax.set_ylabel('Median Yearly Salary (USD)')
plt.suptitle('')
save_fig(fig, 'eda_salary_remote_vs_onsite')
plt.show()

## 4. Posting Duration & Ghost Jobs

**Research Theme 2**: How long do jobs stay open? Long-duration + low-application postings may be "ghost jobs."

In [ ]:
if 'days_open' in df.columns:
    df_duration = df[df['days_open'].notna() & (df['days_open'] >= 0)].copy()
    print(f"Postings with valid days_open: {len(df_duration):,}")
    
    plot_distribution(
        df_duration['days_open'],
        title='Distribution of Posting Duration (Days Open)',
        xlabel='Days Open',
        clip=(0, 180),
        bins=60,
        save_name='eda_days_open_distribution'
    )
    plt.show()
    
    print("\nDays open statistics:")
    print(df_duration['days_open'].describe().round(1))
    
    # Flag potential ghost jobs: open > 60 days with very few applications
    if 'applies' in df_duration.columns:
        ghost_candidates = df_duration[
            (df_duration['days_open'] > 60) & 
            (df_duration['applies'].fillna(0) < 5)
        ]
        print(f"\nPotential ghost jobs (open >60 days, <5 applies): {len(ghost_candidates):,}")
        print(f"  = {len(ghost_candidates)/len(df_duration)*100:.1f}% of postings with duration data")
else:
    print("days_open column not available — check cleaning step.")

## 5. Job Posting Volume Over Time

When were jobs posted? Are there seasonal trends?

In [ ]:
if 'listed_time' in df.columns:
    plot_time_series(
        df['listed_time'],
        title='Weekly Job Posting Volume Over Time',
        ylabel='Number of Postings',
        freq='W',
        save_name='eda_posting_volume_weekly'
    )
    plt.show()
    
    # Monthly view
    plot_time_series(
        df['listed_time'],
        title='Monthly Job Posting Volume',
        ylabel='Number of Postings',
        freq='M',
        save_name='eda_posting_volume_monthly'
    )
    plt.show()

## 6. Application & View Metrics

**Research Theme 4**: What drives engagement? Do remote jobs get more applications?

In [ ]:
# Application rate = applies / views
df_engagement = df[(df['views'].notna()) & (df['views'] > 0)].copy()
df_engagement['application_rate'] = df_engagement['applies'] / df_engagement['views']

print(f"Postings with view data: {len(df_engagement):,}")
print(f"\nApplication rate stats:")
print(df_engagement['application_rate'].describe().round(3))

# Application rate by experience level
print("\nMedian application rate by experience level:")
print(df_engagement.groupby('experience_level')['application_rate'].median().sort_values(ascending=False).round(3))

# Application rate by remote status
print("\nMedian application rate by remote status:")
print(df_engagement.groupby('is_remote')['application_rate'].median().round(3))

In [ ]:
# Views and applies distributions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.histplot(df['views'].dropna().clip(0, 500), bins=50, ax=axes[0])
axes[0].set_title('Distribution of Views (clipped at 500)')
axes[0].set_xlabel('Views')

sns.histplot(df['applies'].dropna().clip(0, 200), bins=50, ax=axes[1])
axes[1].set_title('Distribution of Applications (clipped at 200)')
axes[1].set_xlabel('Applications')

plt.tight_layout()
save_fig(fig, 'eda_views_applies_distribution')
plt.show()

## 7. Location & Work Type

In [ ]:
# Top locations
plot_top_categories(
    df['location'].fillna('Unknown'),
    title='Top 20 Job Locations',
    top_n=20,
    save_name='eda_top_locations'
)
plt.show()

# Work type distribution
if 'formatted_work_type' in df.columns:
    print("\nWork Type Distribution:")
    print(df['formatted_work_type'].value_counts())

## 8. Correlation Analysis

Which numeric features correlate with each other?

In [ ]:
numeric_cols = ['med_salary_yearly', 'min_salary_yearly', 'max_salary_yearly',
                'views', 'applies', 'days_open']
numeric_cols = [c for c in numeric_cols if c in df.columns]

plot_correlation_heatmap(
    df[numeric_cols].dropna(),
    title='Correlation Heatmap of Key Numeric Features',
    save_name='eda_correlation_heatmap'
)
plt.show()

## Key Findings

*(Fill in after running the notebook)*

### Theme 1: Salary
- Median salary: $___
- Salary range by experience level: ...
- Remote salary premium/penalty: ...

### Theme 2: Ghost Jobs
- Median days open: ___
- Potential ghost jobs: ___% of postings

### Theme 3: Entry-Level
- Entry-level postings: ___% of total
- *(Deep analysis in Phase 3)*

### Theme 4: Engagement
- Median application rate: ___
- Remote jobs get more/fewer applications than on-site

---

**→ Next phase**: Phase 2 — NLP Feature Engineering (notebooks 04-06)